# 19 — Capstone: a reproducible digit classification experiment

**Level:** Advanced / expert practice · **Study time:** 120–180 minutes (including experiments)

**Prerequisites:** Notebooks 00–17; notebook 18 is optional for this project.

## What you will be able to do
- Own a complete workflow from split to saved inference artifact.
- Compare configurations on validation data and report a final test result.
- Write an honest model report and design your next experiment.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 18](18_distributed_training.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Define the experiment and reserve a test set

Task: classify a clean 8 × 8 grayscale digit as 0–9. Metric: accuracy plus macro recall. Baseline: the training majority class. Split: 60% train, 20% validation, 20% test with stratification. We scale by the documented fixed pixel range, 0–16. Keep the test set out of configuration and epoch selection.


In [ ]:
import copy
import json
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits = load_digits()
raw = torch.tensor(digits.images, dtype=torch.float32)
X = (raw / 16).unsqueeze(1)
y = torch.tensor(digits.target, dtype=torch.long)
development, test_idx = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y.numpy(), random_state=SEED)
train_idx, val_idx = train_test_split(development, test_size=0.25, stratify=y[development].numpy(), random_state=SEED)
assert set(train_idx).isdisjoint(val_idx) and set(development).isdisjoint(test_idx)
print('Split sizes:', len(train_idx), len(val_idx), len(test_idx))


## Step 3 — Compare two configurations fairly

Each run starts from the same seed and uses a freshly seeded shuffled loader. A scheduler lowers the learning rate after each epoch. Select the best checkpoint by validation cross entropy; report validation accuracy alongside it. We use a fixed small search budget, not a promise that either configuration is optimal.


In [ ]:
def make_model(width):
    return nn.Sequential(nn.Conv2d(1, width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Flatten(), nn.Linear(width * 4 * 4, 10))

configs = [{'width': 8, 'lr': 0.003, 'weight_decay': 0.0},
           {'width': 16, 'lr': 0.003, 'weight_decay': 0.001}]
runs = []
for config in configs:
    torch.manual_seed(SEED)
    model = make_model(config['width'])
    loader = DataLoader(TensorDataset(X[train_idx], y[train_idx]), batch_size=64, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED), num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=18)
    best_val = float('inf')
    curve = []
    for epoch in range(18):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(xb), yb)
            loss.backward(); optimizer.step()
        scheduler.step()
        model.eval()
        with torch.no_grad():
            logits = model(X[val_idx])
            val_loss = nn.functional.cross_entropy(logits, y[val_idx]).item()
            val_acc = (logits.argmax(1) == y[val_idx]).float().mean().item()
        curve.append(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch, best_acc = epoch + 1, val_acc
    runs.append({'config': config, 'state': best_state, 'val_loss': best_val,
                 'val_accuracy': best_acc, 'epoch': best_epoch, 'curve': curve})
    print(config, 'validation accuracy:', best_acc, 'selected epoch:', best_epoch)
selected = min(runs, key=lambda run: run['val_loss'])
final_model = make_model(selected['config']['width'])
final_model.load_state_dict(selected['state']); final_model.eval()
for run in runs:
    plt.plot(run['curve'], label=f"width={run['config']['width']}")
plt.xlabel('Epoch'); plt.ylabel('Validation cross entropy'); plt.legend(); plt.show()


## Step 4 — Evaluate the locked model

Now compute test metrics once. A confusion matrix and example errors help describe limitations. They do not authorize another round of tuning on the same test set. This is an educational estimate on one split; writer identity is not available for a writer-disjoint split here.


In [ ]:
with torch.inference_mode():
    test_probabilities = final_model(X[test_idx]).softmax(1)
test_predictions = test_probabilities.argmax(1)
truth = y[test_idx]
confusion = torch.bincount(truth * 10 + test_predictions, minlength=100).reshape(10, 10)
accuracy = (test_predictions == truth).float().mean().item()
macro_recall = (confusion.diag() / confusion.sum(1).clamp_min(1)).mean().item()
majority = torch.bincount(y[train_idx]).argmax()
baseline_accuracy = (truth == majority).float().mean().item()
metrics = {'test_accuracy': accuracy, 'test_macro_recall': macro_recall,
           'majority_baseline_accuracy': baseline_accuracy, 'test_examples': len(test_idx)}
print(metrics)
assert accuracy > baseline_accuracy + 0.5
plt.imshow(confusion, cmap='Blues'); plt.xlabel('Predicted class'); plt.ylabel('True class'); plt.colorbar(); plt.show()
mistakes = (test_predictions != truth).nonzero().flatten()[:6]
if len(mistakes):
    fig, axes = plt.subplots(1, len(mistakes), figsize=(12, 2), squeeze=False)
    for ax, i in zip(axes[0], mistakes.tolist()):
        ax.imshow(raw[test_idx[i]], cmap='gray')
        ax.set_title(f'{truth[i].item()} -> {test_predictions[i].item()}'); ax.axis('off')
    plt.show()


## Step 5 — Save an artifact and test the inference contract

The prediction function accepts raw images `[B,8,8]` in the 0–16 range. It validates shape, finite values, and range, then applies the saved scale. Reload parity verifies the artifact is useful outside the training function. The checkpoint and JSON report are written under artifacts/19_capstone.


In [ ]:
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks').is_dir() and (p / 'requirements.txt').exists())
output = root / 'artifacts' / '19_capstone'
output.mkdir(parents=True, exist_ok=True)
torch.save({'state': final_model.state_dict(), 'config': selected['config'],
            'pixel_scale': 16., 'classes': list(range(10)), 'seed': SEED}, output / 'digits.pt')
payload = torch.load(output / 'digits.pt', map_location='cpu', weights_only=True)
loaded = make_model(payload['config']['width'])
loaded.load_state_dict(payload['state']); loaded.eval()
def predict_digits(raw_images):
    images = torch.as_tensor(raw_images, dtype=torch.float32, device='cpu')
    if images.ndim != 3 or tuple(images.shape[1:]) != (8, 8):
        raise ValueError('Expected a batch with shape [B, 8, 8].')
    if len(images) == 0 or not torch.isfinite(images).all() or (images < 0).any() or (images > 16).any():
        raise ValueError('Provide a nonempty batch of finite pixels in [0, 16].')
    with torch.inference_mode():
        return loaded((images / payload['pixel_scale']).unsqueeze(1)).softmax(1)
assert torch.allclose(predict_digits(raw[test_idx[:10]]), test_probabilities[:10], atol=1e-6)
report = {'selected_config': selected['config'], 'selected_epoch': selected['epoch'],
          'validation_accuracy': selected['val_accuracy'], **metrics,
          'seed': SEED, 'torch_version': str(torch.__version__),
          'limitations': ['Small low-resolution dataset', 'Single split and seed',
                          'No writer-disjoint evaluation', 'No external or shifted image evaluation']}
(output / 'metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved model and report to:', output)


## Your turn — practice before opening the solutions

1. Verify the inference function rejects an image batch with the wrong shape.
2. Compute test confidence for correct and incorrect predictions, and explain why this is not a calibration test.
3. Write a model report in the journal: task, split, baseline, selection method, metrics, failure modes, and one next experiment. Use the structured reference below as a checklist, not as a claim you have run that next experiment.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
try:
    predict_digits(torch.zeros(4, 64))
except ValueError as error:
    print('Expected validation error:', error)
else:
    raise AssertionError('Bad shape should be rejected.')
confidence = test_probabilities.max(1).values
correct = test_predictions == truth
for name, mask in [('correct', correct), ('incorrect', ~correct)]:
    print(name, 'mean confidence:', confidence[mask].mean().item() if mask.any() else 'no examples')
print(json.dumps(report, indent=2))
# Next experiment proposal: compare across three seeds using validation only,
# then evaluate once on a newly reserved or external writer-disjoint test set.
# Mean confidence by correctness alone does not measure probability calibration.


## Before moving on

Completion checklist: rebuild the training loop without copying; explain every tensor shape; beat a baseline; prevent leakage; justify validation selection; reload the artifact; and describe a failure case. Expert skill comes from repeating this process on unfamiliar data and diagnosing failures, not from finishing a fixed number of notebooks.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [PyTorch quickstart](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)
- [Digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)


[Course guide](../README.md) · [Previous: 18](18_distributed_training.ipynb)
